# NB04 — L2-full: estructura profunda de Attractor Convergence

Ordenado por `main.depth.vsm` (estimador NB03: full stack L1–L5) — **nivel L2
completo**: ① inventario de las 22 ecuaciones numeradas, ② grafo de
dependencias entre envs, ③ auditoría *proof-vs-asserted* de los 17 envs de
teorema. Sin LLM (regex determinista, R10) — la verificación simbólica (L3)
viene en NB06.

Funciones nuevas en `l2_parser`: `equation_inventory`, `dependency_graph`,
`proof_audit`. Tests: `python3 -m pytest test_l2_parser.py -q` → 16 passed.

## ① Inventario de ecuaciones (27 filas = 22 bloques + 5 `eq:` sueltos)

In [ ]:
from pathlib import Path
import sys
from collections import Counter
sys.path.insert(0, str(Path.cwd()))
from l2_parser import equation_inventory, dependency_graph, proof_audit

MAIN = Path("../main.tex")
if not MAIN.exists():
    MAIN = Path("data/main.tex")
tex = MAIN.read_text(errors="replace")

inv = equation_inventory(tex)
print(f"rows={len(inv)} blocks={sum(1 for r in inv if r['block'])} "
      f"loose={sum(1 for r in inv if not r['block'])} labeled={sum(1 for r in inv if r['label'])}")
print(f"{'#':<4}{'line':<7}{'label':<28}section")
for r in inv:
    print(f"{r['idx']:<4}{r['line']:<7}{r['label'] or '—':<28}{r['section'][:44]}")
print("\npor seccion:", dict(Counter(r["section"] for r in inv)))

rows=27 blocks=22 loose=5 labeled=20
#   line   label                       section
1   408    —                           The State Vector
2   460    eq:stasis_cos               Stasis similarity metric
3   477    eq:dynamics                 Dynamics
4   496    eq:softmax_mixing           Dynamics
5   531    eq:kernel                   The dynamics as a Markov chain
6   552    eq:geomergodic              The dynamics as a Markov chain
7   568    eq:pf_matrix                The dynamics as a Markov chain
8   579    eq:pf_lyap                  The dynamics as a Markov chain
9   612    —                           The Feasible Region $\Omega$
10  622    eq:a0                       The Feasible Region $\Omega$
11  624    eq:a01                      The Feasible Region $\Omega$
12  626    eq:a3                       The Feasible Region $\Omega$
13  628    eq:a5                       The Feasible Region $\Omega$
14  630    eq:ae                       The Feasible Region $\Omega$
15  656    —

### Lectura

- **20/27** ecuaciones etiquetadas; 7 bloques sin label (display suelto) —
  candidatos a `eq:` si algún `\ref` futuro los necesita.
- Concentración en **The Feasible Region Ω (7)** y **Markov chain (4)** —
  el núcleo matemático del paper vive ahí; Lyapunov tiene 2 (`eq:lyapunov`,
  `eq:convergence`) — las que Fierro discute.

## ② Grafo de dependencias (envs etiquetados ↔ labels referenciados)

In [ ]:
g = dependency_graph(tex)
d = g["denominator"]
print(f"nodes={d['nodes']} edges={d['edges']} dangling={d['dangling']}")
print("nodos por kind:", dict(Counter(n["kind"] for n in g["nodes"])))
for a, b in g["edges"]:
    print(f"  {a} -> {b}")

nodes=37 edges=16 dangling=0
nodos por kind: {'equation': 20, 'definition': 5, 'remark': 8, 'theorem': 2, 'corollary': 1, 'proposition': 1}
  cor:lyaplandauer -> sec:ly01
  def:kernel -> eq:dynamics
  def:omega_a -> sec:ly01
  rem:emergent_envelope -> sec:ly01
  rem:emergent_envelope -> sec:soplo
  remark@L590 -> eq:dynamics
  remark@L590 -> eq:geomergodic
  remark@L590 -> eq:pf_lyap
  remark@L767 -> sec:ly01
  remark@L767 -> sec:sustained_noise
  remark@L778 -> sec:ly01
  remark@L778 -> thm:hybrid_invariance
  thm:convergence -> eq:dynamics
  thm:hybrid_invariance -> sec:sustained_noise
  thm:hybrid_invariance -> sec:thermal_cycling
  thm:hybrid_invariance -> thm:convergence


### Lectura

- **0 dangling** (los 16 `\ref` resuelven — coherencia B1 confirmada a nivel
  de grafo).
- Cadena central: `thm:hybrid_invariance → thm:convergence → eq:dynamics`
  — la invarancia híbrida descansa en el teorema de convergencia, que
  descansa en la dinámica. Los remarks anclan experimentos
  (`sec:soplo`, `sec:sustained_noise`, `sec:thermal_cycling`) a esta cadena.

## ③ proof-vs-asserted (los 17 envs, denominador completo)

In [ ]:
audit = proof_audit(tex)
c = Counter(r["classification"] for r in audit)
print(f"envs={len(audit)}  {dict(c)}")
print(f"{'kind':<13}{'label':<32}{'class':<26}section")
for r in audit:
    print(f"{r['kind']:<13}{r['label'] or '—':<32}{r['classification']:<26}{r['section'][:38]}")
print("\nglobal 'discharged by measurement' en todo el doc:",
      tex.count("discharged by measurement"))
print("entornos proof en todo el doc:", tex.count("\\begin{proof}"))

envs=17  {'statement-only': 16, 'discharged-by-measurement': 1}
kind         label                         class                      section
definition   def:kernel                    statement-only             The dynamics as a Markov chain
remark       —                             statement-only             The dynamics as a Markov chain
definition   def:omega                     statement-only             The Feasible Region $\Omega$
definition   def:omega_a                   statement-only             The Feasible Region $\Omega$
theorem      thm:hybrid_invariance         discharged-by-measurement  The Feasible Region $\Omega$
remark       —                             statement-only             The Feasible Region $\Omega$
remark       —                             statement-only             The Feasible Region $\Omega$
definition   def:lyapunov                  statement-only             Lyapunov Convergence
theorem      thm:convergence               statement-only             

### Hallazgo D2 — **0 demostraciones formales en todo el paper**

| clase | n | detalle |
|---|---|---|
| `formal-proof` | **0/17** | `\begin{proof}` = 0 en todo el documento |
| `discharged-by-measurement` | 1/17 | solo `thm:hybrid_invariance` (la descarga vía EXP-* está declarada en el propio enunciado) |
| `statement-only` | 16/17 | enunciado sin demostración ni marcador — incluye `thm:convergence`, `cor:lyaplandauer`, `prop:s5_deterministic` |

**Lectura honesta:** el paper es de *ingeniería convalidada por medición*
(experimentos EXP-*) no de matemática con demostraciones — coherente con su
propia frase "discharged by measurement" (×2 globales: 1 dentro de envs, 1 en
prosa). **No es un defecto en sí**: es el tipo de documento. Pero significa
que el nivel L3 (verificación simbólica) es el único que puede aportar
garantías formales reales — exactamente lo que ordena NB06.

*(Una de las 2 ocurrencias globales queda fuera de los envs: la de prosa.)*

In [ ]:
import subprocess
tests = Path.cwd() if (Path.cwd() / "test_l2_parser.py").exists() else next(
    (r for r in [Path.cwd(), *Path.cwd().parents] if (r / "test_l2_parser.py").exists()), None)
if not tests:
    print("tests: en local 16/16 (R1 locality)")
else:
    r = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pytest no disponible en este env — suite 16/16 en local (R1 locality)")
    else:
        r = subprocess.run([sys.executable, "-m", "pytest", "test_l2_parser.py", "-q"],
                           capture_output=True, text=True, cwd=str(tests))
        print("$ pytest test_l2_parser.py -q")
        print("\n".join(r.stdout.strip().splitlines()[-2:]))

$ pytest test_l2_parser.py -q
................                                                         [100%]
16 passed in 0.45s


## Estado y siguiente

- **NB04 ✓** L2-full: inventario + grafo + proof-audit (tests 16/16).
- **NB05** — estudio de grafos de nodos por capas (micro → macro), notebook
  separado (pedido S5).
- **NB06** — L3 verificación simbólica (los ~8 checks: convexidad Ω_A,
  λ=ln(0.99), J=0.99·I, deriva, L_f…) — el nivel que puede certificar lo que
  NB04 marca como *statement-only*.